# Optimal static policy vs. optimal threshold policy (Table 4, Section 8.3)

Two types with delay sensitivities $\theta_1>\theta_2$, Poisson arrivals with rates $\lambda_1,\lambda_2$, exponential service with mean equal to the assigned depth, preemptive priority.

* **Static policy:** depths $(x_1,x_2)$ and a priority order.
* **Threshold policy:** depths $(x_1,x_2)$, a priority order, and a workload threshold $N$. An arrival is admitted iff the expected workload $n_1x_1+n_2x_2$ is below $N$. $N=\infty$ recovers the static policy.
* **Decentralized optimum** in either model: the best policy with $w_1\le w_2$, the incentive-compatibility condition of Section 4.2.

For each ratio $\lambda_2/\lambda_1$ the notebook reports the welfare gap under both policy classes and the welfare gain of threshold control. Runtime is about 10-15 minutes per ratio.

In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

## Instance

In [2]:
THETA1, THETA2 = 0.105, 0.10
LAM = 1.0
RATIOS = [0.5, 1, 2, 4, 8, 16, 32]         # lambda2 / lambda1
V = np.log1p                               # V(x) = log(1 + x). A linear V is degenerate under threshold
                                           # policies (welfare -> a - theta2 as x2 -> inf).
LAM1 = LAM2 = None                         # set per instance in main

## Static model: sojourn times under preemptive priority

In [3]:
def static_sojourn(x1, x2, prio):
    """Expected sojourn times under preemptive priority to type `prio`. None if unstable."""
    r1, r2 = LAM1 * x1, LAM2 * x2
    if r1 + r2 >= 1:
        return None
    if prio == 1:
        w1 = x1 / (1 - r1)
        w2 = x2 / (1 - r1 - r2) + LAM1 * x1**2 / ((1 - r1) * (1 - r1 - r2))
    else:
        w2 = x2 / (1 - r2)
        w1 = x1 / (1 - r1 - r2) + LAM2 * x2**2 / ((1 - r2) * (1 - r1 - r2))
    return w1, w2

def static_welfare(x1, x2, prio):
    w = static_sojourn(x1, x2, prio)
    if w is None:
        return -np.inf, None
    w1, w2 = w
    return LAM1 * (V(x1) - THETA1 * w1) + LAM2 * (V(x2) - THETA2 * w2), (w1, w2)

## Threshold model: exact stationary analysis of the finite chain on $(n_1,n_2)$

Transitions: a type-$i$ arrival moves to $n+e_i$ at rate $\lambda_i$ when $n_1x_1+n_2x_2<N$; the prioritized nonempty class $s$ completes at rate $1/x_s$ and moves to $n-e_s$. The stationary distribution solves $\pi Q=0$, $\sum\pi=1$. By PASTA the admission probability is $\alpha=\sum_{n:\,n_1x_1+n_2x_2<N}\pi(n)$, the same for both types, and by Little's law $w_i=\mathbb{E}[N_i]/(\lambda_i\alpha)$.

In [4]:
def threshold_performance(x1, x2, N, prio):
    """Stationary analysis of (n1, n2) under the threshold policy.

    Returns (welfare, alpha, w1, w2): alpha is the admission probability (same for both types by
    PASTA), w_i the expected sojourn time of admitted type-i customers (Little's law)."""
    n1max = int(N // x1) + 1               # after admitting at workload < N, n1*x1 < N + x1
    n2max = int(N // x2) + 1
    states = [(a, b) for a in range(n1max + 1) for b in range(n2max + 1)]
    index = {s: k for k, s in enumerate(states)}
    S = len(states)
    Q = lil_matrix((S, S))
    for (a, b), k in index.items():
        if a * x1 + b * x2 < N:                            # admissions
            Q[k, index[(a + 1, b)]] += LAM1
            Q[k, index[(a, b + 1)]] += LAM2
        if prio == 1:                                       # service completion of the class in service
            if a > 0:
                Q[k, index[(a - 1, b)]] += 1 / x1
            elif b > 0:
                Q[k, index[(a, b - 1)]] += 1 / x2
        else:
            if b > 0:
                Q[k, index[(a, b - 1)]] += 1 / x2
            elif a > 0:
                Q[k, index[(a - 1, b)]] += 1 / x1
    for k in range(S):
        Q[k, k] = -Q[k, :].sum()
    # stationary distribution: pi Q = 0, sum(pi) = 1 (replace one balance equation by normalization)
    A = Q.T.tolil()
    A[0, :] = 1.0
    rhs = np.zeros(S)
    rhs[0] = 1.0
    pi = spsolve(A.tocsc(), rhs)
    n1 = np.array([s[0] for s in states])
    n2 = np.array([s[1] for s in states])
    admit = n1 * x1 + n2 * x2 < N
    alpha = pi[admit].sum()
    EN1, EN2 = (n1 * pi).sum(), (n2 * pi).sum()
    w1, w2 = EN1 / (LAM1 * alpha), EN2 / (LAM2 * alpha)
    welfare = alpha * (LAM1 * V(x1) + LAM2 * V(x2)) - THETA1 * EN1 - THETA2 * EN2
    return welfare, alpha, w1, w2

def threshold_welfare(x1, x2, N, prio):
    if min(x1, x2, N) <= 1e-3 or (N // x1 + 1) * (N // x2 + 1) > 40000:
        return -np.inf, None
    welfare, alpha, w1, w2 = threshold_performance(x1, x2, N, prio)
    return welfare, (alpha, w1, w2)

## Optimization: Nelder-Mead from a grid of starts, for each priority order

In [5]:
def optimize(objective, starts, ic):
    """Maximize `objective(v, prio)` over v (depths, and N for the threshold model) and priority.
    `objective` returns (welfare, info) with info[-2:] = (w1, w2). If ic, require w1 <= w2."""
    best = (-np.inf, None, None)
    for prio in (1, 2):
        def loss(v):
            welfare, info = objective(np.abs(v), prio)
            if info is None or (ic and info[-2] > info[-1] + 1e-9):
                return 1e9
            return -welfare
        for v0 in starts:
            r = minimize(loss, v0, method="Nelder-Mead", options=dict(xatol=1e-6, fatol=1e-10, maxfev=3000))
            if -r.fun > best[0]:
                best = (-r.fun, np.abs(r.x), prio)
    return best

def solve_static(ic):
    starts = [(a, b) for a in (0.05, 0.3, 1.0, 2.0) for b in (0.05, 0.3, 1.0, 2.0)]
    W, v, prio = optimize(lambda v, p: static_welfare(v[0], v[1], p), starts, ic)
    w1, w2 = static_welfare(v[0], v[1], prio)[1]
    return dict(W=W, x1=v[0], x2=v[1], prio=prio, w1=w1, w2=w2)

def solve_threshold(ic):
    starts = [(a, b, n) for a in (0.1, 0.5, 1.5) for b in (0.1, 0.5, 1.5) for n in (0.3, 1.0, 3.0, 8.0)]
    W, v, prio = optimize(lambda v, p: threshold_welfare(v[0], v[1], v[2], p), starts, ic)
    alpha, w1, w2 = threshold_welfare(v[0], v[1], v[2], prio)[1]
    return dict(W=W, x1=v[0], x2=v[1], N=v[2], prio=prio, alpha=alpha, w1=w1, w2=w2)

## Run

In [6]:
def show(name, r, decentralized):
    extra = f"  N={r['N']:.4f}  alpha={r['alpha']:.4f}" if "N" in r else ""
    print(f"  {name:<26} prio={r['prio']}  x1={r['x1']:.4f}  x2={r['x2']:.4f}{extra}  "
          f"w1={r['w1']:.4f}  w2={r['w2']:.4f}  W={r['W']:.5f}"
          + ("" if decentralized else f"  IC={'yes' if r['w1'] <= r['w2'] + 1e-9 else 'no'}"), flush=True)

print(f"theta = ({THETA1}, {THETA2}), lambda = {LAM}, V(x) = log(1+x)", flush=True)
summary = []
for ratio in RATIOS:
    LAM1, LAM2 = LAM / (1 + ratio), LAM * ratio / (1 + ratio)
    print(f"\nlambda2/lambda1 = {ratio}  (computing, about 10-15 minutes per ratio)", flush=True)
    s, sd = solve_static(ic=False), solve_static(ic=True)
    t, td = solve_threshold(ic=False), solve_threshold(ic=True)
    show("static, centralized", s, False)
    show("static, decentralized", sd, True)
    show("threshold, centralized", t, False)
    show("threshold, decentralized", td, True)
    summary.append((ratio, s["prio"], 100 * (s["W"] - sd["W"]) / s["W"], t["prio"], 100 * (t["W"] - td["W"]) / t["W"],
                    100 * (t["W"] / s["W"] - 1), 100 * (td["W"] / sd["W"] - 1)))

theta = (0.105, 0.1), lambda = 1.0, V(x) = log(1+x)

lambda2/lambda1 = 0.5  (computing, about 10-15 minutes per ratio)
  static, centralized        prio=1  x1=0.4984  x2=0.8772  w1=0.7465  w2=2.9981  W=0.32735  IC=yes
  static, decentralized      prio=1  x1=0.4984  x2=0.8772  w1=0.7465  w2=2.9981  W=0.32735
  threshold, centralized     prio=1  x1=0.8397  x2=1.2192  N=2.4634  alpha=0.7723  w1=1.1580  w2=3.3415  W=0.37046  IC=yes
  threshold, decentralized   prio=1  x1=0.8397  x2=1.2192  N=2.4634  alpha=0.7723  w1=1.1580  w2=3.3415  W=0.37046

lambda2/lambda1 = 1  (computing, about 10-15 minutes per ratio)
  static, centralized        prio=1  x1=0.4760  x2=0.7572  w1=0.6247  w2=2.3632  W=0.32560  IC=yes
  static, decentralized      prio=1  x1=0.4760  x2=0.7572  w1=0.6247  w2=2.3632  W=0.32560
  threshold, centralized     prio=1  x1=0.8165  x2=1.1175  N=2.4087  alpha=0.7698  w1=1.0173  w2=2.7723  W=0.37071  IC=yes
  threshold, decentralized   prio=1  x1=0.8165  x2=1.1175  N=2.2404  alpha=

## Summary

In [7]:
print("\nsummary")
print(f"{'l2/l1':>6} {'static prio':>11} {'static gap':>10} {'thr prio':>8} {'thr gap':>8} {'gain C':>7} {'gain D':>7}")
for ratio, sp, sg, tp, tg, gc, gd in summary:
    print(f"{ratio:6.1f} {sp:>11d} {sg:9.3f}% {tp:>8d} {tg:7.3f}% {gc:6.2f}% {gd:6.2f}%")


summary
 l2/l1 static prio static gap thr prio  thr gap  gain C  gain D
   0.5           1     0.000%        1   0.000%  13.17%  13.17%
   1.0           1     0.000%        1   0.000%  13.85%  13.85%
   2.0           2     0.402%        1   0.000%  14.05%  14.51%
   4.0           2     1.679%        2   0.059%  13.12%  14.98%
   8.0           2     2.198%        2   0.248%  13.02%  15.28%
  16.0           2     2.026%        2   0.238%  13.37%  15.44%
  32.0           2     1.498%        2   0.195%  14.02%  15.53%
